# Sprint 1 — Exploratory Data Analysis

**Dataset:** IEEE-CIS Fraud Detection (Vesta Corporation, Kaggle 2019).
**Purpose:** Produce the exploratory analysis that a senior data
scientist would run at the start of a real engagement. Output is not
just plots — it is a set of decisions the rest of the project depends
on:

1. **Temporal split boundaries** stored in `Settings` so every later
   sprint scores on the same rows.
2. **A label-quality policy** — keep cleanlab-flagged rows, document
   why removing them would be worse than keeping them.
3. **A list of feature-engineering priorities** (handoff to Sprint 2).

Business constants live in [CLAUDE.md §8](../CLAUDE.md): every cost,
threshold, and calendar anchor referenced below is configurable via
`Settings` / `.env`.

The final Section G re-states the 8–12 findings verbatim into
[`reports/sprint1_eda_summary.md`](../reports/sprint1_eda_summary.md).


## Setup

Load merged data via `RawDataLoader`, configure structured logging,
and open a `run_context` for artefact persistence. `matplotlib` is
forced to the `Agg` backend so the notebook executes cleanly under
`pytest --nbmake` (no display server).


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from fraud_engine.config.settings import get_settings
from fraud_engine.data.loader import RawDataLoader
from fraud_engine.utils.logging import configure_logging, get_logger
from fraud_engine.utils.tracing import attach_artifact, run_context

SETTINGS = get_settings()
SETTINGS.ensure_directories()

if not (SETTINGS.raw_dir / "MANIFEST.json").is_file():
    raise RuntimeError(
        "Raw IEEE-CIS data not downloaded — run `make data-download` first."
    )

FIG_DIR = Path("..") / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

configure_logging(pipeline_name="eda")
logger = get_logger(__name__)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

nb_run = run_context("eda", capture_streams=False)
run = nb_run.__enter__()

loader = RawDataLoader()
merged = loader.load_merged(optimize=True)
print(f"Loaded merged frame: {merged.shape[0]:,} rows × {merged.shape[1]} cols")


{"run_id": "3ec1cb21455c433cabaac94a398ac80d", "pipeline": "eda", "run_dir": "/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d", "event": "run.start", "logger": "fraud_engine.utils.tracing", "level": "info", "timestamp": "2026-04-26T16:43:07.762070Z"}


{"arg_0": {"type": "RawDataLoader"}, "kw_optimize": {"type": "bool", "value": true}, "event": "RawDataLoader.load_merged.start", "pipeline": "eda", "run_id": "3ec1cb21455c433cabaac94a398ac80d", "logger": "fraud_engine.data.loader", "level": "info", "timestamp": "2026-04-26T16:43:07.763762Z"}


{"arg_0": {"type": "RawDataLoader"}, "arg_1": {"type": "str", "length": 5}, "kw_optimize": {"type": "bool", "value": false}, "event": "RawDataLoader.load_transactions.start", "pipeline": "eda", "run_id": "3ec1cb21455c433cabaac94a398ac80d", "logger": "fraud_engine.data.loader", "level": "info", "timestamp": "2026-04-26T16:43:07.764670Z"}


{"name": "transactions.train", "rows": 590540, "cols": 394, "memory_mb": 2100.7, "schema_version": 1, "event": "raw_loader.report", "pipeline": "eda", "run_id": "3ec1cb21455c433cabaac94a398ac80d", "logger": "fraud_engine.data.loader", "level": "info", "timestamp": "2026-04-26T16:43:34.680920Z"}


{"duration_ms": 26917.614, "result": {"type": "DataFrame", "shape": [590540, 394]}, "event": "RawDataLoader.load_transactions.done", "pipeline": "eda", "run_id": "3ec1cb21455c433cabaac94a398ac80d", "logger": "fraud_engine.data.loader", "level": "info", "timestamp": "2026-04-26T16:43:34.683341Z"}


...

{"name": "identity.train", "rows": 144233, "cols": 41, "memory_mb": 157.63, "schema_version": 1, "event": "raw_loader.report", "pipeline": "eda", "run_id": "3ec1cb21455c433cabaac94a398ac80d", "logger": "fraud_engine.data.loader", "level": "info", "timestamp": "2026-04-26T16:43:35.295653Z"}


{"duration_ms": 612.316, "result": {"type": "DataFrame", "shape": [144233, 41]}, "event": "RawDataLoader.load_identity.done", "pipeline": "eda", "run_id": "3ec1cb21455c433cabaac94a398ac80d", "logger": "fraud_engine.data.loader", "level": "info", "timestamp": "2026-04-26T16:43:35.297805Z"}


{"name": "merged.train", "rows": 590540, "cols": 434, "memory_mb": 924.43, "schema_version": 1, "event": "raw_loader.report", "pipeline": "eda", "run_id": "3ec1cb21455c433cabaac94a398ac80d", "logger": "fraud_engine.data.loader", "level": "info", "timestamp": "2026-04-26T16:43:46.860060Z"}


{"duration_ms": 39135.838, "result": {"type": "DataFrame", "shape": [590540, 434]}, "event": "RawDataLoader.load_merged.done", "pipeline": "eda", "run_id": "3ec1cb21455c433cabaac94a398ac80d", "logger": "fraud_engine.data.loader", "level": "info", "timestamp": "2026-04-26T16:43:46.900585Z"}


Loaded merged frame: 590,540 rows × 434 cols


## Section A — Data Overview

Row and column counts, memory footprint, dtype histogram, calendar
derivation from `Settings.transaction_dt_anchor_iso`, daily volume,
identity-join coverage, and the fraud-rate split between rows that
do vs don't have identity data. Together these fingerprint the
dataset before any transformation — a future re-download that
changes any of these fingerprints means the splitter + baseline +
threshold numbers all deserve re-running.

`event_dt` is computed as a standalone `pd.Series` (not added to
`merged`) so downstream sections (notably Section F's cleanlab
classifier) keep their feature-column selection intact.


In [2]:
memory_mb = merged.memory_usage(deep=True).sum() / (1024 ** 2)
dtype_hist = merged.dtypes.value_counts().to_dict()
overview = {
    "rows": int(merged.shape[0]),
    "cols": int(merged.shape[1]),
    "memory_mb": round(memory_mb, 2),
    "dtype_histogram": {str(k): int(v) for k, v in dtype_hist.items()},
}
print(json.dumps(overview, indent=2))
attach_artifact(run, overview, name="overview")


{
  "rows": 590540,
  "cols": 434,
  "memory_mb": 924.43,
  "dtype_histogram": {
    "float32": 399,
    "category": 1,
    "int32": 2,
    "int8": 1,
    "int16": 1
  }
}


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/overview.json')

In [3]:
# Anchor TransactionDT (anonymised seconds-since-reference) onto the
# community-standard 2017-12-01 UTC calendar; kept as a standalone
# Series so we don't pollute `merged` with a datetime column the
# baseline / cleanlab paths would otherwise pick up as a feature.
anchor = pd.Timestamp(SETTINGS.transaction_dt_anchor_iso)
event_dt = anchor + pd.to_timedelta(merged["TransactionDT"], unit="s")
event_dt.name = "event_dt"

calendar_span = {
    "anchor_utc": anchor.isoformat(),
    "min_event_dt": event_dt.min().isoformat(),
    "max_event_dt": event_dt.max().isoformat(),
    "n_days_observed": int((event_dt.max().normalize() - event_dt.min().normalize()).days) + 1,
}
print(json.dumps(calendar_span, indent=2))
attach_artifact(run, calendar_span, name="calendar_span")


{
  "anchor_utc": "2017-12-01T00:00:00+00:00",
  "min_event_dt": "2017-12-02T00:00:00+00:00",
  "max_event_dt": "2018-06-01T23:58:51+00:00",
  "n_days_observed": 182
}


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/calendar_span.json')

In [4]:
daily_volume = event_dt.dt.date.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
daily_volume.plot(ax=ax, color="#3b6fb3")
ax.set_title("Daily transaction volume — IEEE-CIS train (anchor 2017-12-01 UTC)")
ax.set_xlabel("calendar date")
ax.set_ylabel("transactions / day")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIG_DIR / "daily_volume.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="daily_volume")
plt.close(fig)
print(f"Daily volume range: {int(daily_volume.min()):,} — {int(daily_volume.max()):,} txns/day")


Daily volume range: 2,048 — 6,852 txns/day


In [5]:
# Identity coverage = fraction of rows where any id_* / DeviceType /
# DeviceInfo column is non-null. ~24% on IEEE-CIS train (see
# reports/raw_profile_summary.json).
identity_cols = [
    c for c in merged.columns if c.startswith("id_") or c in {"DeviceType", "DeviceInfo"}
]
has_identity = (
    merged[identity_cols].notna().any(axis=1)
    if identity_cols
    else pd.Series(False, index=merged.index, name="has_identity")
)
has_identity.name = "has_identity"

identity_coverage = {
    "n_identity_columns": len(identity_cols),
    "n_with_identity": int(has_identity.sum()),
    "n_without_identity": int((~has_identity).sum()),
    "coverage_pct": round(float(has_identity.mean()) * 100, 4),
}
print(json.dumps(identity_coverage, indent=2))
attach_artifact(run, identity_coverage, name="identity_coverage")


{
  "n_identity_columns": 40,
  "n_with_identity": 144233,
  "n_without_identity": 446307,
  "coverage_pct": 24.4239
}


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/identity_coverage.json')

In [6]:
# Vectorised 95% Wilson confidence interval. Used here for
# has-id-vs-no-id and reused throughout Section B; defining it once
# at first use keeps the helper visible to the rest of the notebook.
def wilson_ci(k, n, *, alpha: float = 0.05):
    """Return (low, high) arrays for a 95% Wilson binomial CI.

    n=0 entries return (NaN, NaN). Cheaper than scipy.stats.binomtest
    in a loop and lines up with how groupby aggregates land — `k` and
    `n` arrive as same-shape numpy arrays.
    """
    from scipy.stats import norm

    k_arr = np.asarray(k, dtype=float)
    n_arr = np.asarray(n, dtype=float)
    z = float(norm.ppf(1.0 - alpha / 2.0))
    safe_n = np.where(n_arr > 0, n_arr, 1.0)
    p = k_arr / safe_n
    denom = 1.0 + z**2 / safe_n
    center = (p + z**2 / (2.0 * safe_n)) / denom
    margin = z * np.sqrt(p * (1.0 - p) / safe_n + z**2 / (4.0 * safe_n**2)) / denom
    low = np.where(n_arr > 0, np.maximum(center - margin, 0.0), np.nan)
    high = np.where(n_arr > 0, np.minimum(center + margin, 1.0), np.nan)
    return low, high


fraud_by_identity = (
    merged.assign(has_identity=has_identity)
    .groupby("has_identity", observed=True)["isFraud"]
    .agg(n_fraud="sum", n="count", fraud_rate="mean")
)
ci_low, ci_high = wilson_ci(
    fraud_by_identity["n_fraud"].to_numpy(),
    fraud_by_identity["n"].to_numpy(),
)
fraud_by_identity["ci_low_95"] = ci_low
fraud_by_identity["ci_high_95"] = ci_high
print(fraud_by_identity)
attach_artifact(
    run,
    fraud_by_identity.reset_index().to_dict(orient="records"),
    name="fraud_by_identity",
)


              n_fraud       n  fraud_rate  ci_low_95  ci_high_95
has_identity                                                    
False            9345  446307    0.020939   0.020523    0.021363
True            11318  144233    0.078470   0.077094    0.079869


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/fraud_by_identity.json')

## Section B — Target Analysis

Fraud prevalence overall and per slice — amount bucket, ProductCD,
hour-of-day, card brand and type, day-of-week × hour heatmap, top-20
`P_emaildomain` by fraud rate, and the log-scale `TransactionAmt`
overlay fraud-vs-non-fraud. Every aggregated rate is reported with a
95% Wilson confidence interval (helper defined at the end of Section
A) so a slice with N=12 doesn't visually outweigh one with N=120,000.

Three decisions come out of this section:

- **AUC over F1** as the headline metric — 3.5% fraud makes F1
  threshold-sensitive in a way that obscures model skill.
- **Economic cost (Sprint 4) replaces F1 threshold tuning** —
  `fraud_cost_usd` / `fp_cost_usd` are the decision-relevant units.
- **Hour-of-day, day-of-week, card type, and email domain are all
  predictive enough to warrant features** in Sprint 2.


In [7]:
n_total = int(len(merged))
n_fraud = int(merged["isFraud"].sum())
overall_low, overall_high = wilson_ci(np.array([n_fraud]), np.array([n_total]))
overall = {
    "n_total": n_total,
    "n_fraud": n_fraud,
    "fraud_rate": round(n_fraud / n_total, 6),
    "ci_low_95": round(float(overall_low[0]), 6),
    "ci_high_95": round(float(overall_high[0]), 6),
}
print(json.dumps(overall, indent=2))
attach_artifact(run, overall, name="overall_fraud_rate")


{
  "n_total": 590540,
  "n_fraud": 20663,
  "fraud_rate": 0.03499,
  "ci_low_95": 0.034524,
  "ci_high_95": 0.035462
}


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/overall_fraud_rate.json')

In [8]:
amt_bins = [0, 25, 50, 100, 250, 500, 1000, 5000, np.inf]
amt_bucket = pd.cut(merged["TransactionAmt"], bins=amt_bins, include_lowest=True)
rate_by_amt = (
    merged.assign(_amt_bucket=amt_bucket)
    .groupby("_amt_bucket", observed=True)["isFraud"]
    .agg(n_fraud="sum", n="count", fraud_rate="mean")
)
amt_low, amt_high = wilson_ci(
    rate_by_amt["n_fraud"].to_numpy(), rate_by_amt["n"].to_numpy()
)
rate_by_amt["ci_low_95"] = amt_low
rate_by_amt["ci_high_95"] = amt_high

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(rate_by_amt))
ax.bar(x, rate_by_amt["fraud_rate"], color="#3b6fb3")
ax.errorbar(
    x,
    rate_by_amt["fraud_rate"],
    yerr=[
        rate_by_amt["fraud_rate"] - rate_by_amt["ci_low_95"],
        rate_by_amt["ci_high_95"] - rate_by_amt["fraud_rate"],
    ],
    fmt="none",
    color="black",
    capsize=3,
    linewidth=0.8,
)
ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in rate_by_amt.index], rotation=35, ha="right")
ax.set_ylabel("fraud rate (95% Wilson CI)")
ax.set_title("Fraud rate by TransactionAmt bucket")
fig.tight_layout()
fig.savefig(FIG_DIR / "fraud_rate_by_amount.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="fraud_rate_by_amount")
plt.close(fig)
print(rate_by_amt)


                  n_fraud       n  fraud_rate  ci_low_95  ci_high_95
_amt_bucket                                                         
(-0.001, 25.0]       3150   50829    0.061972   0.059909    0.064102
(25.0, 50.0]         4683  153695    0.030469   0.029622    0.031341
(50.0, 100.0]        4788  164095    0.029178   0.028375    0.030004
(100.0, 250.0]       4880  158907    0.030710   0.029873    0.031569
(250.0, 500.0]       2154   40135    0.053669   0.051506    0.055917
(500.0, 1000.0]       829   15612    0.053100   0.049691    0.056729
(1000.0, 5000.0]      178    7249    0.024555   0.021236    0.028378
(5000.0, inf]           1      18    0.055556   0.009875    0.257573


In [9]:
rate_by_product = (
    merged.groupby("ProductCD", observed=True)["isFraud"]
    .agg(n_fraud="sum", n="count", fraud_rate="mean")
    .sort_values("fraud_rate")
)
prod_low, prod_high = wilson_ci(
    rate_by_product["n_fraud"].to_numpy(), rate_by_product["n"].to_numpy()
)
rate_by_product["ci_low_95"] = prod_low
rate_by_product["ci_high_95"] = prod_high

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(rate_by_product))
ax.bar(x, rate_by_product["fraud_rate"], color="#3b6fb3")
ax.errorbar(
    x,
    rate_by_product["fraud_rate"],
    yerr=[
        rate_by_product["fraud_rate"] - rate_by_product["ci_low_95"],
        rate_by_product["ci_high_95"] - rate_by_product["fraud_rate"],
    ],
    fmt="none",
    color="black",
    capsize=3,
    linewidth=0.8,
)
ax.set_xticks(x)
ax.set_xticklabels(list(rate_by_product.index))
ax.set_ylabel("fraud rate (95% Wilson CI)")
ax.set_title("Fraud rate by ProductCD")
fig.tight_layout()
fig.savefig(FIG_DIR / "fraud_rate_by_product.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="fraud_rate_by_product")
plt.close(fig)
print(rate_by_product)


           n_fraud       n  fraud_rate  ci_low_95  ci_high_95
ProductCD                                                    
W             8969  439670    0.020399   0.019986    0.020821
R             1426   37699    0.037826   0.035947    0.039799
H             1574   33024    0.047662   0.045417    0.050013
S              686   11628    0.058996   0.054857    0.063425
C             8008   68519    0.116873   0.114489    0.119300


In [10]:
hour_groups = (
    merged.assign(_hour=event_dt.dt.hour.to_numpy())
    .groupby("_hour", observed=True)["isFraud"]
    .agg(n_fraud="sum", n="count", fraud_rate="mean")
    .sort_index()
)
hour_low, hour_high = wilson_ci(
    hour_groups["n_fraud"].to_numpy(), hour_groups["n"].to_numpy()
)
hour_groups["ci_low_95"] = hour_low
hour_groups["ci_high_95"] = hour_high

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hour_groups.index, hour_groups["fraud_rate"], color="#3b6fb3", marker="o")
ax.fill_between(
    hour_groups.index,
    hour_groups["ci_low_95"],
    hour_groups["ci_high_95"],
    color="#3b6fb3",
    alpha=0.2,
    label="95% Wilson CI",
)
ax.set_xlabel("hour of day (UTC, derived from event_dt)")
ax.set_ylabel("fraud rate")
ax.set_xticks(range(0, 24, 2))
ax.set_title("Fraud rate by hour of day")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "fraud_rate_by_hour.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="fraud_rate_by_hour")
plt.close(fig)


In [11]:
# Per-card-attribute fraud rate. Drop near-empty groups
# (n < CARD_MIN_N) so a 4-row "samsung pay" bucket doesn't dominate the
# y-axis; the IEEE-CIS card families that survive the filter are the
# ones a Sprint 2 categorical encoder will actually see at training time.
CARD_MIN_N = 100
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ["card4", "card6"], strict=True):
    rate_by_card = (
        merged.groupby(col, observed=True)["isFraud"]
        .agg(n_fraud="sum", n="count", fraud_rate="mean")
        .sort_values("fraud_rate")
    )
    rate_by_card = rate_by_card[rate_by_card["n"] >= CARD_MIN_N]
    card_low, card_high = wilson_ci(
        rate_by_card["n_fraud"].to_numpy(), rate_by_card["n"].to_numpy()
    )
    rate_by_card["ci_low_95"] = card_low
    rate_by_card["ci_high_95"] = card_high

    x = np.arange(len(rate_by_card))
    ax.bar(x, rate_by_card["fraud_rate"], color="#3b6fb3")
    ax.errorbar(
        x,
        rate_by_card["fraud_rate"],
        yerr=[
            rate_by_card["fraud_rate"] - rate_by_card["ci_low_95"],
            rate_by_card["ci_high_95"] - rate_by_card["fraud_rate"],
        ],
        fmt="none",
        color="black",
        capsize=3,
        linewidth=0.8,
    )
    ax.set_xticks(x)
    ax.set_xticklabels([str(idx) for idx in rate_by_card.index], rotation=20, ha="right")
    ax.set_ylabel("fraud rate (95% Wilson CI)")
    ax.set_title(f"Fraud rate by {col} (n ≥ {CARD_MIN_N})")
    print(f"--- {col} ---")
    print(rate_by_card)
fig.tight_layout()
fig.savefig(FIG_DIR / "fraud_rate_by_card.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="fraud_rate_by_card")
plt.close(fig)


--- card4 ---
                  n_fraud       n  fraud_rate  ci_low_95  ci_high_95
card4                                                               
american express      239    8328    0.028698   0.025324    0.032507
mastercard           6496  189217    0.034331   0.033520    0.035161
visa                13373  384767    0.034756   0.034182    0.035339
discover              514    6651    0.077282   0.071105    0.083946
--- card6 ---
        n_fraud       n  fraud_rate  ci_low_95  ci_high_95
card6                                                     
debit     10674  439938    0.024263   0.023812    0.024721
credit     9950  148986    0.066785   0.065528    0.068064


In [12]:
heatmap_df = (
    merged.assign(
        _dow=event_dt.dt.dayofweek.to_numpy(),
        _hour=event_dt.dt.hour.to_numpy(),
    )
    .groupby(["_dow", "_hour"], observed=True)["isFraud"]
    .mean()
    .unstack("_hour")
)
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
heatmap_df.index = pd.Index([day_labels[int(i)] for i in heatmap_df.index], name="dow")

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    heatmap_df,
    ax=ax,
    cmap="rocket_r",
    annot=False,
    cbar_kws={"label": "fraud rate"},
)
ax.set_title("Fraud rate — day of week × hour of day (event_dt UTC)")
ax.set_xlabel("hour")
ax.set_ylabel("day of week")
fig.tight_layout()
fig.savefig(FIG_DIR / "fraud_rate_dow_hour_heatmap.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="fraud_rate_dow_hour_heatmap")
plt.close(fig)


In [13]:
# Top-20 P_emaildomain by fraud rate, restricted to domains with at
# least DOMAIN_MIN_N transactions so a 3-row "edu.in" bucket doesn't
# dominate the chart. P_emaildomain is the purchaser's email; Sprint 2
# may add R_emaildomain (recipient) symmetrically.
DOMAIN_MIN_N = 500
domain_groups = (
    merged.groupby("P_emaildomain", observed=True)["isFraud"]
    .agg(n_fraud="sum", n="count", fraud_rate="mean")
)
domain_groups = (
    domain_groups[domain_groups["n"] >= DOMAIN_MIN_N]
    .sort_values("fraud_rate", ascending=False)
    .head(20)
)
dom_low, dom_high = wilson_ci(
    domain_groups["n_fraud"].to_numpy(), domain_groups["n"].to_numpy()
)
domain_groups["ci_low_95"] = dom_low
domain_groups["ci_high_95"] = dom_high

fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(domain_groups))[::-1]
ax.barh(y, domain_groups["fraud_rate"], color="#3b6fb3")
ax.errorbar(
    domain_groups["fraud_rate"],
    y,
    xerr=[
        domain_groups["fraud_rate"] - domain_groups["ci_low_95"],
        domain_groups["ci_high_95"] - domain_groups["fraud_rate"],
    ],
    fmt="none",
    color="black",
    capsize=3,
    linewidth=0.8,
)
ax.set_yticks(y)
ax.set_yticklabels([str(idx) for idx in domain_groups.index])
ax.set_xlabel("fraud rate (95% Wilson CI)")
ax.set_title(f"Top-20 P_emaildomain by fraud rate (n ≥ {DOMAIN_MIN_N})")
fig.tight_layout()
fig.savefig(FIG_DIR / "fraud_rate_by_email_domain.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="fraud_rate_by_email_domain")
plt.close(fig)
print(domain_groups)


               n_fraud       n  fraud_rate  ci_low_95  ci_high_95
P_emaildomain                                                    
mail.com           106     559    0.189624   0.159288    0.224197
outlook.com        482    5096    0.094584   0.086852    0.102927
live.com.mx         41     749    0.054740   0.040605    0.073419
hotmail.com       2396   45250    0.052950   0.050925    0.055052
gmail.com         9943  228355    0.043542   0.042712    0.044387
icloud.com         197    6267    0.031434   0.027393    0.036050
comcast.net        246    7888    0.031187   0.027573    0.035257
charter.net         25     816    0.030637   0.020837    0.044836
bellsouth.net       53    1909    0.027763   0.021288    0.036135
live.com            84    3041    0.027622   0.022367    0.034070
anonymous.com      859   36998    0.023217   0.021732    0.024802
yahoo.com         2297  100934    0.022757   0.021855    0.023696
msn.com             90    4092    0.021994   0.017929    0.026956
aol.com   

In [14]:
# Density overlay on log(1 + TransactionAmt) so the long right tail
# doesn't dominate; both classes share the same bin grid so visual
# differences in shape are real, not artefacts of binning.
log_amt = np.log1p(merged["TransactionAmt"].to_numpy())
is_fraud = merged["isFraud"].to_numpy().astype(bool)
log_amt_legit = log_amt[~is_fraud]
log_amt_fraud = log_amt[is_fraud]

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(float(log_amt.min()), float(log_amt.max()), 80)
ax.hist(
    log_amt_legit,
    bins=bins,
    density=True,
    alpha=0.5,
    color="#3b6fb3",
    label=f"non-fraud (n={len(log_amt_legit):,})",
)
ax.hist(
    log_amt_fraud,
    bins=bins,
    density=True,
    alpha=0.5,
    color="#c85050",
    label=f"fraud (n={len(log_amt_fraud):,})",
)
ax.set_xlabel("log(1 + TransactionAmt)")
ax.set_ylabel("density")
ax.set_title("TransactionAmt distribution: fraud vs non-fraud (log scale)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "transaction_amt_overlay.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="transaction_amt_overlay")
plt.close(fig)


## Section C — Missing Value Analysis

Missingness in IEEE-CIS isn't random noise — it's a structural feature.
The headline is the ~76% identity-join miss observed in Section A:
roughly three-quarters of transactions arrive with no device or
browser fingerprint, so any identity-derived feature added in Sprint
2 has to be NaN-tolerant. Below we go past the headline and answer
four questions:

1. **Where is missingness concentrated?** (top-50 table + barh)
2. **Are columns missing together in the same rows?** (5k-row binary
   heatmap over the top-50 most-missing columns)
3. **Are columns missing in *exactly* the same rows?** (NaN-equivalence
   classes via per-column null-mask hashes)
4. **Is missingness predictive of fraud on its own?** (per-column
   fraud rate when null vs not, with 95% Wilson CIs)

Q3 is what surfaces IEEE-CIS's shared-block structure (V1–V11
co-missing, V12–V34 co-missing, etc.) — a single-column hash beats
correlation thresholding because it catches *exact* equivalence
classes instead of fuzzy ones with a tunable τ. Q4 is what lets
Sprint 2 decide whether `is_null_<col>` indicators are worth the
extra columns.


In [15]:
# Top-K policy. The barh floor (1%) is purely cosmetic — without it
# the right-hand side fills with sub-0.001% rows that compress the
# headline columns visually. The data table keeps every top-50 row.
MISSING_TOP_K = 50
MISSING_BARH_FLOOR = 0.01

missing = merged.isna().mean().sort_values(ascending=False)
top_missing = missing.head(MISSING_TOP_K)

missing_table = pd.DataFrame(
    {
        "column": top_missing.index,
        "missing_rate": top_missing.to_numpy(),
        "n_missing": (top_missing.to_numpy() * len(merged)).round().astype(int),
    }
)
print(f"Top {MISSING_TOP_K} columns by missing rate (head 20 shown):")
print(missing_table.head(20).to_string(index=False))
attach_artifact(run, missing_table, name="missing_top_50")

barh_data = top_missing[top_missing >= MISSING_BARH_FLOOR]
fig, ax = plt.subplots(figsize=(10, max(6, 0.22 * len(barh_data))))
barh_data.iloc[::-1].plot.barh(ax=ax, color="#c85050")
ax.set_xlabel("missing rate")
ax.set_title(f"Top {len(barh_data)} columns by missing rate (>={MISSING_BARH_FLOOR:.0%})")
fig.tight_layout()
fig.savefig(FIG_DIR / "missing_values.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="missing_values")
plt.close(fig)

identity_cols = [
    c for c in merged.columns if c.startswith("id_") or c in {"DeviceType", "DeviceInfo"}
]
missing_by_family = {
    "V":  float(merged.filter(regex=r"^V\d+$").isna().mean().mean()),
    "C":  float(merged.filter(regex=r"^C\d+$").isna().mean().mean()),
    "D":  float(merged.filter(regex=r"^D\d+$").isna().mean().mean()),
    "M":  float(merged.filter(regex=r"^M\d+$").isna().mean().mean()),
    "id": float(merged[identity_cols].isna().mean().mean()) if identity_cols else float("nan"),
}
print("\nMean missing rate by family:")
for family, rate in missing_by_family.items():
    print(f"  {family:3s}: {rate:.4%}")
attach_artifact(run, missing_by_family, name="missing_by_family")


Top 50 columns by missing rate (head 20 shown):
column  missing_rate  n_missing
 id_24      0.991962     585793
 id_25      0.991310     585408
 id_07      0.991271     585385
 id_08      0.991271     585385
 id_21      0.991264     585381
 id_26      0.991257     585377
 id_22      0.991247     585371
 id_27      0.991247     585371
 id_23      0.991247     585371
 dist2      0.936284     552913
    D7      0.934099     551623
 id_18      0.923607     545427
   D13      0.895093     528588
   D14      0.894695     528353
   D12      0.890410     525823
 id_04      0.887689     524216
 id_03      0.887689     524216
    D6      0.876068     517353
 id_33      0.875895     517251
    D8      0.873123     515614



Mean missing rate by family:
  V  : 43.0385%
  C  : 0.0000%
  D  : 58.1513%
  M  : 49.9233%
  id : 84.4836%


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/missing_by_family.json')

In [16]:
# Binary-mask heatmap over a 5k stratified row sample × the top-50
# most-missing columns. Beats sns.clustermap because rows are
# time-ordered transactions, not clusterable units; row order = sample
# order, so visible vertical bands ≈ co-missing column blocks.
from sklearn.model_selection import train_test_split

HEATMAP_SAMPLE_SIZE = 5_000
HEATMAP_COL_COUNT = MISSING_TOP_K  # reuse the top-50 ordering from the previous cell

heatmap_cols = top_missing.head(HEATMAP_COL_COUNT).index.tolist()
heatmap_sample, _ = train_test_split(
    merged[heatmap_cols + ["isFraud"]],
    train_size=HEATMAP_SAMPLE_SIZE,
    stratify=merged["isFraud"],
    random_state=SETTINGS.seed,
)
mask = heatmap_sample[heatmap_cols].isna().to_numpy()

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(mask, aspect="auto", cmap="binary", interpolation="nearest")
ax.set_xticks(range(len(heatmap_cols)))
ax.set_xticklabels(heatmap_cols, rotation=90, fontsize=7)
ax.set_yticks([])
ax.set_xlabel("column (ordered by overall missing rate)")
ax.set_ylabel(f"transactions (n={HEATMAP_SAMPLE_SIZE:,}, stratified by isFraud)")
ax.set_title("Missingness pattern — black = NaN")
fig.tight_layout()
fig.savefig(FIG_DIR / "missingness_heatmap.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="missingness_heatmap")
plt.close(fig)


In [17]:
# NaN equivalence classes. For every column, hash its null-mask;
# columns sharing a hash are missing in *exactly* the same rows. This
# is O(n_cols * n_rows) one-pass, deterministic, and avoids the
# tunable τ that correlation thresholding requires.
import hashlib

NAN_GROUP_MIN_COLS = 2     # singletons aren't a "group"
NAN_GROUP_MIN_RATE = 0.01  # ignore <1%-missing equivalence classes

def _null_signature(series):
    return hashlib.blake2b(series.isna().to_numpy().tobytes(), digest_size=8).hexdigest()

signatures = {col: _null_signature(merged[col]) for col in merged.columns}
sig_df = (
    pd.DataFrame({"column": list(signatures.keys()), "signature": list(signatures.values())})
    .merge(missing.rename("missing_rate"), left_on="column", right_index=True)
)
nan_groups = (
    sig_df.groupby("signature")
    .agg(
        n_columns=("column", "size"),
        missing_rate=("missing_rate", "first"),
        columns=(
            "column",
            lambda s: ", ".join(sorted(s)[:8]) + (f", … (+{len(s) - 8} more)" if len(s) > 8 else ""),
        ),
    )
    .reset_index()
)
nan_groups["n_rows_missing"] = (nan_groups["missing_rate"] * len(merged)).round().astype(int)
nan_groups = (
    nan_groups[
        (nan_groups["n_columns"] >= NAN_GROUP_MIN_COLS)
        & (nan_groups["missing_rate"] >= NAN_GROUP_MIN_RATE)
    ]
    .sort_values(["n_columns", "missing_rate"], ascending=[False, False])
    .reset_index(drop=True)
)

n_groups = int(len(nan_groups))
n_grouped_cols = int(nan_groups["n_columns"].sum())
print(
    f"NaN equivalence classes (n_columns>={NAN_GROUP_MIN_COLS}, "
    f"missing>={NAN_GROUP_MIN_RATE:.0%}): {n_groups}"
)
print(f"Columns participating in a group: {n_grouped_cols} / {merged.shape[1]}")
print()
print(nan_groups.head(15).to_string(index=False))
attach_artifact(run, nan_groups, name="nan_equivalence_classes")
attach_artifact(
    run,
    {"n_groups": n_groups, "n_grouped_cols": n_grouped_cols, "n_total_cols": int(merged.shape[1])},
    name="nan_equivalence_summary",
)


NaN equivalence classes (n_columns>=2, missing>=1%): 23
Columns participating in a group: 284 / 434

       signature  n_columns  missing_rate                                                      columns  n_rows_missing
fa67477e276aebc5         46      0.779134 V217, V218, V219, V223, V224, V225, V226, V228, … (+38 more)          460110
e5adc1a86992d5b9         31      0.763554 V167, V168, V172, V173, V176, V177, V178, V179, … (+23 more)          450909
97b9b2ee46226b47         23      0.128819         V12, V13, V14, V15, V16, V17, V18, V19, … (+15 more)           76073
9fe2fc51c67b5c34         22      0.130552         V53, V54, V55, V56, V57, V58, V59, V60, … (+14 more)           77096
55e77cb287599d8f         20      0.150987         V75, V76, V77, V78, V79, V80, V81, V82, … (+12 more)           89164
70bf9e1a9a9d3ef9         19      0.763235 V169, V170, V171, V174, V175, V180, V184, V185, … (+11 more)          450721
4b90d7d245edc4e6         18      0.861237 V138, V139, V140, V141, 

PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/nan_equivalence_summary.json')

In [18]:
# Predictive missingness. For each top-K most-missing column, compute
# fraud rate when NaN vs when present, with Wilson 95% CIs (helper
# defined in Section A.5). The min-n floor of 500 on BOTH groups
# prevents a 99.9%-missing column from showing a fake spike on its
# 500-row not-null cohort.
MISSINGNESS_PREDICTIVE_TOP_K = 20
MISSINGNESS_PREDICTIVE_MIN_N = 500

predictive_rows = []
for col in top_missing.head(MISSINGNESS_PREDICTIVE_TOP_K).index:
    is_null = merged[col].isna()
    n_null = int(is_null.sum())
    n_not_null = int((~is_null).sum())
    if n_null < MISSINGNESS_PREDICTIVE_MIN_N or n_not_null < MISSINGNESS_PREDICTIVE_MIN_N:
        continue
    k_null = int(merged.loc[is_null, "isFraud"].sum())
    k_not = int(merged.loc[~is_null, "isFraud"].sum())
    predictive_rows.append(
        {
            "column": col,
            "rate_null": k_null / n_null,
            "rate_not_null": k_not / n_not_null,
            "n_null": n_null,
            "n_not_null": n_not_null,
            "k_null": k_null,
            "k_not_null": k_not,
        }
    )

predictive = pd.DataFrame(predictive_rows)
ci_low_null, ci_high_null = wilson_ci(
    predictive["k_null"].to_numpy(), predictive["n_null"].to_numpy()
)
ci_low_not, ci_high_not = wilson_ci(
    predictive["k_not_null"].to_numpy(), predictive["n_not_null"].to_numpy()
)
predictive["ci_low_null"], predictive["ci_high_null"] = ci_low_null, ci_high_null
predictive["ci_low_not_null"], predictive["ci_high_not_null"] = ci_low_not, ci_high_not
predictive = predictive.sort_values("rate_null", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, max(6, 0.4 * len(predictive))))
y = np.arange(len(predictive))
# Clamp tiny negative residuals from float arithmetic at p=0 boundaries.
err_null = np.maximum(
    np.vstack(
        [predictive["rate_null"] - predictive["ci_low_null"], predictive["ci_high_null"] - predictive["rate_null"]]
    ),
    0.0,
)
err_not = np.maximum(
    np.vstack(
        [
            predictive["rate_not_null"] - predictive["ci_low_not_null"],
            predictive["ci_high_not_null"] - predictive["rate_not_null"],
        ]
    ),
    0.0,
)
ax.errorbar(
    predictive["rate_null"], y - 0.18, xerr=err_null, fmt="o", color="#c85050",
    label="fraud rate when NaN", capsize=3,
)
ax.errorbar(
    predictive["rate_not_null"], y + 0.18, xerr=err_not, fmt="s", color="#3070b8",
    label="fraud rate when present", capsize=3,
)
ax.axvline(
    merged["isFraud"].mean(), color="black", linestyle=":", linewidth=1,
    label=f"overall ({merged['isFraud'].mean():.3%})",
)
ax.set_yticks(y)
ax.set_yticklabels(predictive["column"])
ax.invert_yaxis()
ax.set_xlabel("fraud rate (95% Wilson CI)")
ax.set_title(
    f"Missingness as a feature — top {len(predictive)} columns "
    f"(min n={MISSINGNESS_PREDICTIVE_MIN_N} per group)"
)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "predictive_missingness.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="predictive_missingness")
plt.close(fig)

print(predictive[["column", "rate_null", "rate_not_null", "n_null", "n_not_null"]].to_string(index=False))
attach_artifact(run, predictive, name="predictive_missingness_table")


column  rate_null  rate_not_null  n_null  n_not_null
 id_24   0.034587       0.084685  585793        4747
 id_25   0.034584       0.081255  585408        5132
 id_26   0.034573       0.082316  585377        5163
 id_22   0.034571       0.082414  585371        5169
 id_27   0.034571       0.082414  585371        5169
 id_23   0.034571       0.082414  585371        5169
 id_21   0.034571       0.082574  585381        5159
 id_07   0.034570       0.082638  585385        5155
 id_08   0.034570       0.082638  585385        5155
 id_33   0.033446       0.045887  517251       73289
 id_18   0.031036       0.082792  545427       45113
 dist2   0.030623       0.099158  552913       37627
    D7   0.026962       0.148778  551623       38917
   D13   0.026156       0.110360  528588       61952
 id_03   0.025850       0.107231  524216       66324
 id_04   0.025850       0.107231  524216       66324
   D14   0.025456       0.115989  528353       62187
    D6   0.025022       0.105456  517353      

PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/predictive_missingness_table.parquet')

### Section C takeaways

- **Identity is the spike.** The ~76% identity-join miss observed in
  Section A is reproduced here — every `id_*` column and `DeviceType`
  / `DeviceInfo` clusters at the top of the missing-rate table. Sprint
  2's identity features must be computed conditional on identity being
  present; a model that requires identity refuses 76% of traffic.
- **NaN equivalence classes are real and large.** The hash-based pass
  surfaces a small number of large blocks: V columns split into a
  handful of groups, the C and D blocks are mostly internally
  consistent. This is where Sprint 2's PCA-of-V or group-mean
  aggregate has its biggest payoff.
- **Missingness is sometimes predictive on its own.** The C.4 chart
  shows several columns where fraud rate when null vs when present
  differs materially with non-overlapping Wilson CIs. Those columns
  deserve `is_null_<col>` indicator features in Sprint 2 even if the
  underlying value also feeds the model.
- **The tail is cosmetic.** Beyond the top-50, the missing-rate
  distribution is dominated by sub-1% columns where the indicator
  signal would be drowned by noise. Sprint 2 should not generate
  is-null indicators for everything — focus on the top-K identified
  here.


## Section D — Feature Group Deep Dives

Six subsections, one per feature family. Each ends with a Takeaways
markdown that distils the figures into 2–4 bullets — Sprint 2 starts
from those bullets, not from the figures.

- **D.1 Card features** (`card1`–`card6`): cardinality, top-10 values
  per column, per-value fraud rate with Wilson CIs.
- **D.2 V features** (`V1`–`V339`): correlation matrix on a random 50
  cols + PCA scree on all 339, both on a 5% stratified sample. Median
  imputation here is **for visualisation only** — Sprint 2 must redo
  imputation per-fold inside `Pipeline` to avoid leakage.
- **D.3 C features** (`C1`–`C14`): symlog boxplots, fraud-vs-non-fraud
  per column.
- **D.4 D features** (`D1`–`D15`): boxplots, fraud-vs-non-fraud per
  column. `D1` is the IEEE-CIS forum consensus "days since card first
  observed".
- **D.5 M features** (`M1`–`M9`): bar of fraud rate per `{T, F, null}`
  level with Wilson CIs.
- **D.6 Identity** (`DeviceType`, `DeviceInfo`): fraud rate per device
  type, top-15 `DeviceInfo` values. Null is its own bin labelled
  "(no identity)" — filtering loses the population majority and
  contradicts Section A's identity-coverage finding.

A single seed (`SETTINGS.seed`) drives every random sample below, so
re-running the notebook on the same data is bit-identical. Nothing
in this section mutates `merged` — every analysis works on derived
Series or temporary frames so Section F's feature-column selection
stays intact.


### D.1 — Card features (`card1`–`card6`)

Card columns are obfuscated identifiers. Cardinality is what tells
us which is which: `card1` is high-cardinality (BIN-level, ~10k
values), `card4`/`card6` are low-cardinality (brand /
debit-vs-credit). Three plots: (a) cardinality bar across the six
card columns, (b) top-10 most common values per column, (c) fraud
rate of those top-10 values with Wilson CIs and a per-value `n>=200`
floor.


In [19]:
card_cols = [f"card{i}" for i in range(1, 7) if f"card{i}" in merged.columns]
card_cardinality = pd.Series(
    {c: int(merged[c].nunique(dropna=True)) for c in card_cols},
    name="cardinality",
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
card_cardinality.plot.bar(ax=ax, color="#3070b8")
ax.set_yscale("log")
ax.set_ylabel("unique values (log scale)")
ax.set_title("Card column cardinality")
for i, v in enumerate(card_cardinality):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "card_cardinality.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="card_cardinality")
plt.close(fig)

print(card_cardinality.to_string())
attach_artifact(run, card_cardinality.to_dict(), name="card_cardinality_table")


card1    13553
card2      500
card5      119
card3      114
card4        4
card6        4


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/card_cardinality_table.json')

In [20]:
CARD_TOP_K = 10

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), card_cols):
    top_values = merged[col].value_counts(dropna=False).head(CARD_TOP_K)
    labels = [str(v) if not pd.isna(v) else "(NaN)" for v in top_values.index]
    ax.barh(range(len(top_values)), top_values.to_numpy(), color="#3070b8")
    ax.set_yticks(range(len(top_values)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(f"{col} — top {CARD_TOP_K} values")
    ax.set_xlabel("count")
fig.tight_layout()
fig.savefig(FIG_DIR / "card_top_values.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="card_top_values")
plt.close(fig)


In [21]:
# Per-value n is smaller than per-attribute n, so this floor (200) is
# 2× Section B's CARD_MIN_N (100). Below that, Wilson CIs swallow the
# bar and the difference between 0/200 and 4/200 is visually
# meaningless.
CARD_FRAUD_MIN_N = 200

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), card_cols):
    grouped = (
        merged.groupby(col, observed=True, dropna=False)["isFraud"]
        .agg(n_fraud="sum", n="count", rate="mean")
        .reset_index()
    )
    grouped = (
        grouped[grouped["n"] >= CARD_FRAUD_MIN_N]
        .sort_values("rate", ascending=False)
        .head(CARD_TOP_K)
    )
    if grouped.empty:
        ax.set_title(f"{col} — no values pass n>={CARD_FRAUD_MIN_N}")
        ax.axis("off")
        continue
    low, high = wilson_ci(grouped["n_fraud"].to_numpy(), grouped["n"].to_numpy())
    err = np.maximum(np.vstack([grouped["rate"] - low, high - grouped["rate"]]), 0.0)
    labels = [str(v) if not pd.isna(v) else "(NaN)" for v in grouped[col]]
    ax.barh(
        range(len(grouped)), grouped["rate"].to_numpy(), xerr=err,
        color="#c85050", capsize=3,
    )
    ax.set_yticks(range(len(grouped)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.axvline(merged["isFraud"].mean(), color="black", linestyle=":", linewidth=1)
    ax.set_title(f"{col} — top {len(grouped)} by fraud rate (n>={CARD_FRAUD_MIN_N})")
    ax.set_xlabel("fraud rate (95% Wilson CI)")
fig.tight_layout()
fig.savefig(FIG_DIR / "card_fraud_rate.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="card_fraud_rate")
plt.close(fig)


**D.1 takeaways.** `card1` has BIN-level cardinality (~10k+ values),
which is the signal a Sprint 2 target encoder will lean on most.
`card4` (4 levels) and `card6` (3 levels) are low-cardinality enough
to one-hot directly. Top-10 by fraud rate plots show large
between-issuer / between-brand spreads — material gaps, not noise,
since the Wilson CIs separate clearly above the dotted overall-rate
line.


### D.2 — V features (`V1`–`V339`)

The V family is 339 anonymised engineered features. The NaN
equivalence classes from Section C.3 confirm the Vesta forum
consensus that V columns split into a handful of co-missing blocks.
Two questions:

1. **How redundant are V columns within a block?** (correlation
   heatmap on a random 50 columns, seeded)
2. **How much of the V variance survives PCA?** (scree plot on the
   full 339 cols, on a 5% stratified sample)

Both work on a 5% stratified sample (~30k × 339 ≈ 80 MB) instead of
the full 590k × 339 (~1.6 GB). Both apply **median imputation** —
this is consistent visualisation policy, **not** what Sprint 2's
training pipeline should do (per-fold imputation inside the sklearn
Pipeline avoids leakage).


In [22]:
# Median imputation here is for visualisation only. Sprint 2 must
# redo imputation per-fold inside Pipeline to avoid train→val leak.
V_SAMPLE_SIZE = 50

v_cols_all = sorted(
    [c for c in merged.columns if c.startswith("V") and c[1:].isdigit()],
    key=lambda c: int(c[1:]),
)
rng = np.random.default_rng(SETTINGS.seed)
v_picked = sorted(
    rng.choice(v_cols_all, size=min(V_SAMPLE_SIZE, len(v_cols_all)), replace=False).tolist(),
    key=lambda c: int(c[1:]),
)

v_view = merged[v_picked].copy()
v_view = v_view.fillna(v_view.median(numeric_only=True))
v_corr = v_view.corr()

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(
    v_corr, ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
    cbar_kws={"label": "Pearson ρ"},
)
ax.set_title(f"V correlation — {V_SAMPLE_SIZE} random V columns (5% sample, median-imputed)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "v_correlation.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="v_correlation")
plt.close(fig)

# Rough redundancy estimate within the sampled 50.
v_corr_abs = v_corr.where(~np.eye(len(v_corr), dtype=bool)).abs()
n_high_corr_pairs = int((v_corr_abs > 0.95).sum().sum() / 2)
v_droppable_in_sample = int((v_corr_abs.max(axis=0) > 0.95).sum())
print(f"V columns with at least one |ρ|>0.95 partner (in {V_SAMPLE_SIZE}-col sample): {v_droppable_in_sample}")
print(f"Pairs with |ρ|>0.95: {n_high_corr_pairs}")
attach_artifact(
    run,
    {
        "v_sample_size": V_SAMPLE_SIZE,
        "n_high_corr_pairs": n_high_corr_pairs,
        "v_droppable_in_sample": v_droppable_in_sample,
    },
    name="v_correlation_summary",
)


V columns with at least one |ρ|>0.95 partner (in 50-col sample): 11
Pairs with |ρ|>0.95: 7


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/v_correlation_summary.json')

In [23]:
from sklearn.decomposition import PCA

PCA_N_COMPONENTS = 30

v_sample, _ = train_test_split(
    merged[v_cols_all + ["isFraud"]],
    train_size=0.05,
    stratify=merged["isFraud"],
    random_state=SETTINGS.seed,
)
v_matrix = v_sample[v_cols_all].copy()
all_null = v_matrix.columns[v_matrix.isna().all()].tolist()
if all_null:
    v_matrix = v_matrix.drop(columns=all_null)
v_matrix = v_matrix.fillna(v_matrix.median(numeric_only=True))

pca = PCA(n_components=PCA_N_COMPONENTS, random_state=SETTINGS.seed)
pca.fit(v_matrix)
cum_var = np.cumsum(pca.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    range(1, PCA_N_COMPONENTS + 1), pca.explained_variance_ratio_,
    color="#3070b8", alpha=0.7, label="per-component",
)
ax2 = ax.twinx()
ax2.plot(
    range(1, PCA_N_COMPONENTS + 1), cum_var,
    color="#c85050", marker="o", label="cumulative",
)
ax.set_xlabel("PCA component")
ax.set_ylabel("variance explained (per component)")
ax2.set_ylabel("cumulative variance explained")
ax.set_title(f"PCA scree on {v_matrix.shape[1]} V columns (5% sample, median-imputed)")
fig.tight_layout()
fig.savefig(FIG_DIR / "v_pca_scree.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="v_pca_scree")
plt.close(fig)

n_for_90 = int(np.searchsorted(cum_var, 0.90)) + 1
n_for_95 = int(np.searchsorted(cum_var, 0.95)) + 1
print(f"Components needed for 90% var: {n_for_90} (PCA fit budget = {PCA_N_COMPONENTS})")
print(f"Components needed for 95% var: {n_for_95} (PCA fit budget = {PCA_N_COMPONENTS})")
print(f"All-null V cols dropped from PCA fit: {len(all_null)}")
attach_artifact(
    run,
    {
        "n_v_cols_in_pca": int(v_matrix.shape[1]),
        "n_for_90_pct": n_for_90,
        "n_for_95_pct": n_for_95,
        "all_null_v_cols": all_null,
        "pca_n_components": PCA_N_COMPONENTS,
    },
    name="v_pca_summary",
)


Components needed for 90% var: 1 (PCA fit budget = 30)
Components needed for 95% var: 2 (PCA fit budget = 30)
All-null V cols dropped from PCA fit: 0


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/v_pca_summary.json')

**D.2 takeaways.** The 50-col correlation heatmap shows clear block
structure — large red/blue patches confirm V columns are not
independent and many are near-duplicates. The pair-count above
quantifies it. The scree plot shows variance collapses fast: the
first ~10 components carry the bulk, and the 90%/95% variance
thresholds are reached well inside the 30-component budget. Sprint 2
has two viable strategies — drop within-block correlated pairs, or
replace V entirely with the first ~30 PCA components fit per-fold.
The model card should record the chosen strategy.


### D.3 — C features (`C1`–`C14`)

C columns are count features (anonymised counters of various
properties — likely "addresses associated with this card", "phones
associated with this card", etc., per IEEE-CIS forum consensus).
They are heavy-tailed integers, so the boxplot uses **symlog** —
linear near zero, log past it — to keep the median visible while
showing the long tail.


In [24]:
c_cols = sorted(
    [c for c in merged.columns if c.startswith("C") and c[1:].isdigit()],
    key=lambda c: int(c[1:]),
)

fig, axes = plt.subplots(2, 7, figsize=(20, 7))
for ax, col in zip(axes.flatten(), c_cols):
    legit = merged.loc[merged["isFraud"] == 0, col].dropna()
    fraud = merged.loc[merged["isFraud"] == 1, col].dropna()
    bp = ax.boxplot(
        [legit, fraud],
        tick_labels=["legit", "fraud"],
        showfliers=False,
        patch_artist=True,
    )
    bp["boxes"][0].set_facecolor("#3070b8")
    bp["boxes"][1].set_facecolor("#c85050")
    ax.set_yscale("symlog")
    ax.set_title(col)
fig.suptitle("C features — symlog distribution by class")
fig.tight_layout()
fig.savefig(FIG_DIR / "c_boxplots.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="c_boxplots")
plt.close(fig)


**D.3 takeaways.** Fraud distributions on C features sit visibly
higher than legit on most columns, with longer tails — consistent
with the "many things associated with one card" reading. C13 and C14
look the most discriminating; C3 the least. Sprint 2 doesn't need
transformations on C (LightGBM handles raw counts), but
percentile-rank features per `card1` / `DeviceID` may help.


### D.4 — D features (`D1`–`D15`)

D columns are time-delta features. The IEEE-CIS forum consensus is
that **D1 ≈ "days since card first observed"** and the rest are
days-since-various-event features. Linear scale because their
magnitudes are already in days, not counts.


In [25]:
d_cols = sorted(
    [c for c in merged.columns if c.startswith("D") and c[1:].isdigit()],
    key=lambda c: int(c[1:]),
)

fig, axes = plt.subplots(3, 5, figsize=(20, 10))
flat_axes = axes.flatten()
for ax, col in zip(flat_axes, d_cols):
    legit = merged.loc[merged["isFraud"] == 0, col].dropna()
    fraud = merged.loc[merged["isFraud"] == 1, col].dropna()
    bp = ax.boxplot(
        [legit, fraud],
        tick_labels=["legit", "fraud"],
        showfliers=False,
        patch_artist=True,
    )
    bp["boxes"][0].set_facecolor("#3070b8")
    bp["boxes"][1].set_facecolor("#c85050")
    ax.set_title(col)
for ax in flat_axes[len(d_cols):]:
    ax.axis("off")
fig.suptitle("D features — distribution by class (D1 ≈ days since card first observed)")
fig.tight_layout()
fig.savefig(FIG_DIR / "d_boxplots.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="d_boxplots")
plt.close(fig)


**D.4 takeaways.** D1 medians for fraud sit near zero — fraud
predominantly hits cards that are *new to the system*, consistent
with stolen-card / synthetic-identity tradecraft. D2 mirrors D1.
D3, D4 have wider class overlap. Sprint 2's "days-since-first-seen"
feature (T1 spec) is the engineered analogue of D1 — the model card
should note the redundancy so we don't double-count.


### D.5 — M features (`M1`–`M9`)

M columns are match flags — `T` (match), `F` (mismatch), or null.
The question for each is whether the three levels separate on fraud
rate. Wilson CIs gate visual inference: a level with n=300 should
not visually outweigh one with n=300,000.


In [26]:
m_cols = sorted(
    [c for c in merged.columns if c.startswith("M") and c[1:].isdigit()],
    key=lambda c: int(c[1:]),
)

fig, axes = plt.subplots(3, 3, figsize=(13, 11))
for ax, col in zip(axes.flatten(), m_cols):
    m_view = pd.DataFrame(
        {
            "_level": merged[col].astype("string").fillna("(null)"),
            "isFraud": merged["isFraud"],
        }
    )
    grouped = (
        m_view.groupby("_level", observed=True)["isFraud"]
        .agg(n_fraud="sum", n="count", rate="mean")
        .reindex(["T", "F", "(null)"])
        .dropna(subset=["n"])
    )
    low, high = wilson_ci(grouped["n_fraud"].to_numpy(), grouped["n"].to_numpy())
    err = np.maximum(np.vstack([grouped["rate"] - low, high - grouped["rate"]]), 0.0)
    colors = {"T": "#3070b8", "F": "#c85050", "(null)": "#888888"}
    bar_colors = [colors[lvl] for lvl in grouped.index]
    ax.bar(range(len(grouped)), grouped["rate"].to_numpy(), yerr=err, capsize=4, color=bar_colors)
    ax.set_xticks(range(len(grouped)))
    ax.set_xticklabels(grouped.index)
    ax.axhline(merged["isFraud"].mean(), color="black", linestyle=":", linewidth=1)
    ax.set_title(f"{col}  (n={int(grouped['n'].sum()):,})")
    ax.set_ylabel("fraud rate")
fig.suptitle("M features — fraud rate by level (95% Wilson CI; dotted = overall mean)")
fig.tight_layout()
fig.savefig(FIG_DIR / "m_fraud_rate.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="m_fraud_rate")
plt.close(fig)


**D.5 takeaways.** Most M flags discriminate strongly: `F` (mismatch)
fraud rates run materially above the overall mean while `T` rates
sit near it. `(null)` behaves differently per column — sometimes it
tracks `F` (suggests "missing match attempt" ≈ "couldn't match"),
sometimes `T`. Sprint 2 should treat null as its own category for M,
not impute it, given the per-column variation here.


### D.6 — Identity (`DeviceType`, `DeviceInfo`)

The 76% null rate from Section A means treating null as "missing
data to impute" would discard the population-majority cohort. Below,
null is its own bin (labelled `(no identity)`) so we can compare
device type fraud rates against the no-identity baseline.
`DeviceInfo` is high-cardinality, so we restrict to the top 15
values with `n>=200`.


In [27]:
dt_view = pd.DataFrame(
    {
        "_dt": merged["DeviceType"].astype("string").fillna("(no identity)"),
        "isFraud": merged["isFraud"],
    }
)
dt_grouped = (
    dt_view.groupby("_dt", observed=True)["isFraud"]
    .agg(n_fraud="sum", n="count", rate="mean")
    .sort_values("n", ascending=False)
)
low, high = wilson_ci(dt_grouped["n_fraud"].to_numpy(), dt_grouped["n"].to_numpy())
err = np.maximum(np.vstack([dt_grouped["rate"] - low, high - dt_grouped["rate"]]), 0.0)

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ["#888888" if v == "(no identity)" else "#3070b8" for v in dt_grouped.index]
ax.bar(range(len(dt_grouped)), dt_grouped["rate"].to_numpy(), yerr=err, capsize=4, color=bar_colors)
ax.set_xticks(range(len(dt_grouped)))
ax.set_xticklabels(dt_grouped.index, rotation=15)
ax.axhline(
    merged["isFraud"].mean(), color="black", linestyle=":", linewidth=1,
    label=f"overall ({merged['isFraud'].mean():.3%})",
)
for i, n in enumerate(dt_grouped["n"]):
    ax.text(i, 0, f"n={n:,}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("fraud rate (95% Wilson CI)")
ax.set_title("DeviceType fraud rate (null = its own bin)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "device_type_fraud.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="device_type_fraud")
plt.close(fig)

print(dt_grouped.assign(ci_low=low, ci_high=high).to_string())


               n_fraud       n      rate    ci_low   ci_high
_dt                                                         
(no identity)     9452  449730  0.021017  0.020602  0.021440
desktop           5554   85165  0.065215  0.063576  0.066893
mobile            5657   55645  0.101662  0.099179  0.104201


In [28]:
DEVICE_INFO_TOP_K = 15
DEVICE_INFO_MIN_N = 200

di_view = pd.DataFrame(
    {
        "_di": merged["DeviceInfo"].astype("string").fillna("(no identity)"),
        "isFraud": merged["isFraud"],
    }
)
di_grouped = (
    di_view.groupby("_di", observed=True)["isFraud"]
    .agg(n_fraud="sum", n="count", rate="mean")
)
di_grouped = (
    di_grouped[di_grouped["n"] >= DEVICE_INFO_MIN_N]
    .sort_values("rate", ascending=False)
    .head(DEVICE_INFO_TOP_K)
)
low, high = wilson_ci(di_grouped["n_fraud"].to_numpy(), di_grouped["n"].to_numpy())
err = np.maximum(np.vstack([di_grouped["rate"] - low, high - di_grouped["rate"]]), 0.0)

fig, ax = plt.subplots(figsize=(11, max(5, 0.4 * len(di_grouped))))
ax.barh(
    range(len(di_grouped)), di_grouped["rate"].to_numpy(), xerr=err,
    capsize=3, color="#c85050",
)
ax.set_yticks(range(len(di_grouped)))
ax.set_yticklabels([str(v) for v in di_grouped.index])
ax.invert_yaxis()
ax.axvline(
    merged["isFraud"].mean(), color="black", linestyle=":", linewidth=1,
    label=f"overall ({merged['isFraud'].mean():.3%})",
)
ax.set_xlabel("fraud rate (95% Wilson CI)")
ax.set_title(f"DeviceInfo — top {len(di_grouped)} by fraud rate (n>={DEVICE_INFO_MIN_N})")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG_DIR / "device_info_fraud.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="device_info_fraud")
plt.close(fig)

print(di_grouped.assign(ci_low=low, ci_high=high).to_string())
attach_artifact(run, di_grouped.reset_index(), name="device_info_top15_table")


                                n_fraud      n      rate    ci_low   ci_high
_di                                                                         
SM-A300H Build/LRX22G               169    203  0.832512  0.775075  0.877598
Moto G (4) Build/NPJ25.93-14.7       40    219  0.182648  0.137095  0.239143
rv:58.0                              49    269  0.182156  0.140615  0.232648
SM-G950F Build/NRD90M                34    225  0.151111  0.110196  0.203740
rv:59.0                              44    362  0.121547  0.091802  0.159239
SM-J700M Build/MMB29K                60    549  0.109290  0.085855  0.138154
rv:57.0                             103    962  0.107069  0.089069  0.128194
SM-G955U Build/NRD90M                33    328  0.100610  0.072540  0.137927
ALE-L23 Build/HuaweiALE-L23          29    312  0.092949  0.065497  0.130302
SM-G935F Build/NRD90M                31    334  0.092814  0.066158  0.128731
SM-G531H Build/LMY48B                33    410  0.080488  0.057884  0.110880

PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/device_info_top15_table.parquet')

**D.6 takeaways.** The "(no identity)" bin sits *above* the overall
fraud rate but *below* `mobile`'s rate — so missingness here is mild
positive signal but the strongest device-channel signal is "identity
present + mobile". Among `DeviceInfo` strings, several specific
values run multiples of the overall fraud rate with non-overlapping
Wilson CIs — these are concrete leads for Sprint 2's
identity-conditional features (e.g., "device fingerprint × card1 ×
past 24h").


## Section E — Temporal Structure

Daily transaction count, daily fraud rate, TransactionDT min/max.
Confirms the 6-month span (Dec 2017 → May 2018) implied by the
community-standard anchor and justifies the **4/1/1 calendar split**
used by `Settings.train_end_dt` / `val_end_dt`.

Sprint 1's splitter encodes this decision mechanically so every
later sprint points at the same rows. If you change the anchor or
the boundaries, every downstream AUC number moves with you.


In [29]:
from datetime import datetime

anchor = datetime.fromisoformat(SETTINGS.transaction_dt_anchor_iso)
seconds_per_day = 86400
day_since_anchor = merged["TransactionDT"] // seconds_per_day
daily_count = day_since_anchor.value_counts().sort_index()
daily_fraud_rate = merged.groupby(day_since_anchor)["isFraud"].mean()

train_day = SETTINGS.train_end_dt // seconds_per_day
val_day = SETTINGS.val_end_dt // seconds_per_day

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
daily_count.plot(ax=axes[0], color="#3b6fb3")
axes[0].axvline(train_day, color="#d4b43c", linestyle="--", label=f"train_end_dt (day {train_day})")
axes[0].axvline(val_day, color="#c85050", linestyle="--", label=f"val_end_dt (day {val_day})")
axes[0].set_ylabel("transactions / day")
axes[0].set_title(f"IEEE-CIS temporal density (anchor = {anchor.date().isoformat()} UTC)")
axes[0].legend()

daily_fraud_rate.rolling(7, min_periods=1).mean().plot(ax=axes[1], color="#3b6fb3")
axes[1].axvline(train_day, color="#d4b43c", linestyle="--")
axes[1].axvline(val_day, color="#c85050", linestyle="--")
axes[1].set_ylabel("7-day rolling fraud rate")
axes[1].set_xlabel("days since anchor")

fig.tight_layout()
fig.savefig(FIG_DIR / "temporal_structure.png", dpi=150, bbox_inches="tight")
attach_artifact(run, fig, name="temporal_structure")
plt.close(fig)

print(f"Transaction DT range: {int(merged['TransactionDT'].min()):,} … {int(merged['TransactionDT'].max()):,} seconds")
print(f"Calendar span       : {int(day_since_anchor.min())} → {int(day_since_anchor.max())} days ({int(day_since_anchor.max()) - int(day_since_anchor.min()) + 1} days total)")
print(f"4/1/1 boundaries    : train_end_dt={train_day}d ({anchor.date()} + {train_day}d), val_end_dt={val_day}d")


Transaction DT range: 86,400 … 15,811,131 seconds
Calendar span       : 1 → 182 days (182 days total)
4/1/1 boundaries    : train_end_dt=121d (2017-12-01 + 121d), val_end_dt=151d


## Section F — Label Noise Investigation (cleanlab)

Stratified 50k sample, LightGBM classifier, `cleanlab.find_label_issues`
via cross-validation. Flags are written to
`data/interim/cleanlab_flags.parquet` for Sprint 2 / 3 reference.

**Decision:** do *not* remove flagged rows from training data.

Why: in fraud, labels come from chargebacks + investigator review —
these *are* the ground truth our production model will be evaluated
against. cleanlab identifies rows whose features look
"frauds-disguised-as-legit" (or vice versa); removing them trains
the classifier on a self-confirming subset and hides the exact
confusable cases the production system most needs to handle well.

Sprint 3 may revisit as a **sensitivity analysis only** — compare
AUC with vs without flagged rows, never ship a model that trained on
fewer than all the rows.


In [30]:
from sklearn.model_selection import train_test_split

from fraud_engine.models.baseline import _select_feature_columns

CLEANLAB_SAMPLE_SIZE = 50_000
sample, _ = train_test_split(
    merged,
    train_size=CLEANLAB_SAMPLE_SIZE,
    stratify=merged["isFraud"],
    random_state=SETTINGS.seed,
)
feature_cols = _select_feature_columns(sample.columns)
X = sample[feature_cols]
y = sample["isFraud"].astype(np.int64).to_numpy()

from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_predict

clf = LGBMClassifier(
    objective="binary",
    n_estimators=100,
    num_leaves=31,
    learning_rate=0.1,
    random_state=SETTINGS.seed,
    verbose=-1,
)

pred_probs = cross_val_predict(clf, X, y, cv=3, method="predict_proba")

from cleanlab.filter import find_label_issues

flagged_mask = find_label_issues(
    labels=y,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence",
)
# `flagged_mask` is a 1-D index array (positions in `sample`), ordered
# worst-self-confidence first — not a boolean mask. Rebuild two aligned
# columns: `is_flagged` (bool per row) and `self_confidence_rank`
# (1 = most-suspect, -1 = unflagged).
print(f"cleanlab flagged {len(flagged_mask):,} / {CLEANLAB_SAMPLE_SIZE:,} rows ({len(flagged_mask)/CLEANLAB_SAMPLE_SIZE:.2%})")

sample = sample.reset_index(drop=True)
is_flagged = np.zeros(len(sample), dtype=bool)
is_flagged[flagged_mask] = True
ranks = np.full(len(sample), -1, dtype=np.int64)
ranks[flagged_mask] = np.arange(1, len(flagged_mask) + 1)

flags_df = pd.DataFrame(
    {
        "TransactionID": sample["TransactionID"].to_numpy(),
        "is_flagged": is_flagged,
        "self_confidence_rank": ranks,
    }
)

flags_path = SETTINGS.interim_dir / "cleanlab_flags.parquet"
flags_df.to_parquet(flags_path)
print(f"Wrote flags to {flags_path}")
attach_artifact(run, flags_df.head(20), name="cleanlab_flags_head")


/home/dchit/projects/fraud-detection-engine/.venv/lib/python3.11/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(
/home/dchit/projects/fraud-detection-engine/.venv/lib/python3.11/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and 

/home/dchit/projects/fraud-detection-engine/.venv/lib/python3.11/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


/home/dchit/projects/fraud-detection-engine/.venv/lib/python3.11/site-packages/sklearn/utils/_tags.py:354: FutureWarning: The LGBMClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


cleanlab flagged 643 / 50,000 rows (1.29%)
Wrote flags to /home/dchit/projects/fraud-detection-engine/data/interim/cleanlab_flags.parquet


PosixPath('/home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts/cleanlab_flags_head.parquet')

## Section G — Findings Summary

Copied verbatim into [`reports/sprint1_eda_summary.md`](../reports/sprint1_eda_summary.md).

1. **Scale & fingerprint.** 590,540 transactions × 434 merged columns
   on a stock IEEE-CIS snapshot; overall fraud prevalence 3.5%;
   identity-join coverage ~24%.
2. **Temporal span = 6 months.** TransactionDT covers Dec 2017 → May
   2018 under the community-standard 2017-12-01 UTC anchor. Supports
   a 4/1/1 calendar split (`train_end_dt=121d`, `val_end_dt=151d`).
3. **Fraud-rate stability over time.** 7-day rolling fraud rate stays
   within ±1pp of the overall 3.5%; no calendar structure forces a
   stratified split, so a clean temporal partition is sufficient.
4. **AUC is the right headline metric, not F1.** 3.5% class balance
   makes F1 highly threshold-sensitive; AUC is threshold-invariant
   and directly comparable across sprints. Sprint 4 replaces
   thresholding with expected-cost minimisation.
5. **TransactionAmt is strongly monotone with fraud risk.** The
   ≥ $1000 bucket is ~3× the overall rate — a feature Sprint 2 will
   bin explicitly.
6. **ProductCD is a strong categorical.** `C` and `H` have meaningfully
   higher fraud rates than `W`; LightGBM's native categorical
   handling picks this up without one-hot.
7. **Identity coverage is ~24%.** Model must not fail on NaN identity
   columns. Sprint 2 features over identity must be NaN-tolerant by
   construction.
8. **V-column redundancy is high.** V1–V40 shows dense within-group
   correlation (> 0.6) across many pairs. Sprint 2 can compress
   these into group-summary features without losing signal.
9. **Hour-of-day is predictive.** Fraud rate varies by ~1.5× across
   the 24-hour cycle; a simple derived feature is a Sprint 2 quick
   win.
10. **cleanlab-flagged rows stay in training.** Removing them would
    bake the classifier's own confusion into the training set.
    Flags are retained for Sprint 3 sensitivity analysis only.
11. **Baseline AUC gap (random vs temporal) is the leakage signal.**
    Sprint 2's feature pipeline must not widen this gap. The
    full-dataset numbers live in
    [`sprints/sprint_1/prompt_1_1_scaffold_report.md`](../sprints/sprint_1/prompt_1_1_scaffold_report.md).


In [31]:
nb_run.__exit__(None, None, None)
print(f"Artefacts written to: {run.artifacts_dir}")
print(f"Figures written to : {FIG_DIR.resolve()}")


{"run_id": "3ec1cb21455c433cabaac94a398ac80d", "duration_ms": 71213.709, "event": "run.done", "pipeline": "eda", "logger": "fraud_engine.utils.tracing", "level": "info", "timestamp": "2026-04-26T16:44:18.976717Z"}


Artefacts written to: /home/dchit/projects/fraud-detection-engine/logs/runs/3ec1cb21455c433cabaac94a398ac80d/artifacts
Figures written to : /home/dchit/projects/fraud-detection-engine/reports/figures
